# Colab — USAD baseline

1. Upload **`usad_colab.zip`** to Drive `AI-TDP` (windows folder already there).  
2. Enable **GPU**.  
3. Mount Drive → **Run all**.  
4. Results: `MyDrive/AI-TDP/baselines/outputs/usad_concat/`

Guide: `baselines/docs/COLAB_USAD_GUIDE.md`

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Paths, install, unzip code (windows already on Drive)

In [ ]:
from pathlib import Path
import sys, zipfile, subprocess

WINDOWS_DIR = Path("/content/drive/MyDrive/AI-TDP/windows")
USAD_ZIP = Path("/content/drive/MyDrive/AI-TDP/usad_colab.zip")
CODE_ZIP = Path("/content/drive/MyDrive/AI-TDP/baselines_code.zip")
REPO_ROOT = Path("/content/AI-TDP")
OUT_ROOT = Path("/content/drive/MyDrive/AI-TDP/baselines/outputs")
MODE = "concat"

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "numpy", "scikit-learn", "matplotlib",
])
REPO_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO_ROOT / "baselines" / "models" / "usad.py").is_file():
    if USAD_ZIP.is_file():
        print(f"Extracting {USAD_ZIP} ...")
        with zipfile.ZipFile(USAD_ZIP, "r") as zf:
            zf.extractall(REPO_ROOT)
    elif CODE_ZIP.is_file():
        print(f"Extracting {CODE_ZIP} ...")
        with zipfile.ZipFile(CODE_ZIP, "r") as zf:
            zf.extractall(REPO_ROOT)
    else:
        raise FileNotFoundError(
            f"Upload usad_colab.zip to {USAD_ZIP.parent}"
        )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

assert (WINDOWS_DIR / "train.npz").is_file(), f"Expected {WINDOWS_DIR}/train.npz"
assert (WINDOWS_DIR / "val.npz").is_file()
assert (WINDOWS_DIR / "test.npz").is_file()
assert (REPO_ROOT / "baselines" / "train" / "train_usad.py").is_file()

print("Ready — using existing Drive windows")
print("WINDOWS_DIR", WINDOWS_DIR)
print("OUT_ROOT", OUT_ROOT)

## 3. Train USAD

Writes to `OUT_ROOT/usad_concat/`.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import runpy

OUT_DIR = OUT_ROOT / f"usad_{MODE}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.argv = [
    "train_usad.py",
    "--mode", MODE,
    "--windows-dir", str(WINDOWS_DIR),
    "--out-dir", str(OUT_DIR),
    "--epochs", "50",
    "--patience", "8",
    "--batch-size", "32",
    "--lr", "0.001",
]
runpy.run_path(
    str(REPO_ROOT / "baselines" / "train" / "train_usad.py"),
    run_name="__main__",
)

## 4. Plot training curves

In [ ]:
import json
import matplotlib.pyplot as plt

OUT_DIR = OUT_ROOT / f"usad_{MODE}"
history = json.loads((OUT_DIR / "history.json").read_text())
config = json.loads((OUT_DIR / "config.json").read_text())

train_l = history["history"]["train_loss"]
val_l = history["history"]["val_loss"]

plt.figure(figsize=(7, 4))
plt.plot(train_l, label="train")
plt.plot(val_l, label="val_score")
plt.axvline(history["best_epoch"] - 1, color="gray", ls="--", label=f"best epoch {history['best_epoch']}")
plt.xlabel("epoch")
plt.ylabel("loss / score")
plt.title("USAD concat")
plt.legend()
plt.tight_layout()
plt.show()

print("best_val_loss:", config.get("best_val_loss"))
print("test_score:", config.get("test_score"))
print("out_dir:", config.get("out_dir"))

## 5. Evaluate

In [ ]:
import runpy

EVAL_DIR = OUT_ROOT / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

sys.argv = [
    "run_eval.py",
    "--model-id", f"usad_{MODE}",
    "--outputs-root", str(OUT_ROOT),
    "--eval-dir", str(EVAL_DIR),
    "--quantile", "0.95",
]
runpy.run_path(
    str(REPO_ROOT / "baselines" / "eval" / "run_eval.py"),
    run_name="__main__",
)

cmp = EVAL_DIR / "comparison.md"
if cmp.is_file():
    print(cmp.read_text())

## Done

Check Drive: `MyDrive/AI-TDP/baselines/outputs/usad_concat/best.pt`